In [ ]:
from typing import Callable
from pydantic import BaseModel, field_validator, ValidationError
import re
import warnings
warnings.filterwarnings('ignore')

class TwitterPostModel(BaseModel):
    username: str
    post: str

    @field_validator('username')
    def validate_twitter_username(cls, v):
        if not v.startswith('@'):
            raise ValueError("Username must start with '@'")
        if len(v) < 2 or len(v) > 16:
            raise ValueError("Username must include '@' and not more than 15 valid characters")
        if not re.match(r'^@[A-Za-z0-9_]+$', v):
            raise ValueError("Username must consist of only letters, numbers, and underscores after your '@'")
        return v

    @field_validator('post')
    def validate_twitter_post(cls, v):
        if not v or len(v) > 280:
            raise ValueError("Post must not be empty and atmost 280 characters")
        return v


def twitter_post_validator(func: Callable[[str, str], None]) -> Callable[[str, str], None]:
    def wrapper(username: str, post: str):
        try:
            TwitterPostModel(username=username, post=post)
        except ValidationError as e:
            raise ValueError(f"Invalid Twitter data: {e}")
        return func(username, post)
    return wrapper


@twitter_post_validator
def print_twitter_post(username: str, post: str):
    print(f"Valid Twitter username: {username}")
    print(f"Post: {post}")


if __name__ == "__main__":
    try:
        print("Successful post")
        print_twitter_post("@Alfamado", "Welcome to GEN AI FELLOWSHIP can't wait to congratulate you as an AI Engineer at the end of your.")
    except ValueError as e:
        print("Error:", e)
        
        
    print("\nFailed post")
    try:
        print_twitter_post("Damilola", "fail to log you in because username lacks @ or has invalid chars, try again qnd enter valid input")
    except ValueError as e:
        print("Error:", e)